# 01 — EDA : Dataset OCT + Fundus Glaucome
Exploration complète des paires Fundus + OCT avec labels des 4 ophtalmologistes.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
from collections import Counter

# ── Chemins ────────────────────────────────────────────────────────────────
DATA_ROOT  = '/kaggle/input/datasets/marwanenasri/data-on-oct-and-fundus-images-zip/Data on OCT and Fundus Images'
EXCEL_PATH = os.path.join(DATA_ROOT, 'Patient Record-final.xlsx')

print('✅ Paths configured')
print(f'   Data root    : {DATA_ROOT}')
print(f'   Excel exists : {os.path.exists(EXCEL_PATH)}')

patients = sorted([f for f in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, f))])
print(f'\n📁 Dossiers patients : {len(patients)}')
print(patients)

In [ ]:
# ── Lecture Excel ──────────────────────────────────────────────────────────
df_raw = pd.read_excel(EXCEL_PATH, header=1)
print(f'📊 Shape : {df_raw.shape}')
print(f'Colonnes : {df_raw.columns.tolist()}')
df_raw.head(5)

In [ ]:
# ── Nettoyage colonnes ─────────────────────────────────────────────────────
df = df_raw.copy()
df.columns = [str(c).strip() for c in df.columns]

# Colonnes Glaucome des 4 ophtalmologistes (index 2, 4, 6, 8)
glaucoma_cols = [df.columns[2], df.columns[4], df.columns[6], df.columns[8]]
image_col     = df.columns[0]
subject_col   = df.columns[10] if len(df.columns) > 10 else None

print('Image col    :', image_col)
print('Glaucoma cols:', glaucoma_cols)
print('Subject col  :', subject_col)

for col in glaucoma_cols:
    print(f'  {col} → valeurs uniques : {df[col].dropna().unique()}')

In [ ]:
# ── Vote majoritaire des 4 ophtalmos ──────────────────────────────────────
def majority_vote(row):
    votes = []
    for col in glaucoma_cols:
        val = str(row[col]).lower().strip()
        if val in ['yes', 'oui', '1']:
            votes.append(1)
        elif val in ['no', 'non', '0']:
            votes.append(0)
        elif 'suspect' in val:
            votes.append(0.5)
    if not votes:
        return np.nan
    return 1 if np.mean(votes) >= 0.5 else 0

df['label']      = df.apply(majority_vote, axis=1)
df['label_name'] = df['label'].map({1: 'Glaucoma', 0: 'Normal'})
df = df.dropna(subset=['label']).reset_index(drop=True)

print('✅ Labels créés')
print(df['label_name'].value_counts())
print(f'Total : {len(df)} entrées')

In [ ]:
# ── Construction des paires Fundus + OCT ──────────────────────────────────
# Construire un index de tous les fichiers disponibles
all_files = {}
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        all_files[f] = os.path.join(root, f)

print(f'Total fichiers indexés : {len(all_files)}')

records = []
missing = []

for _, row in df.iterrows():
    # Nom de l'image fundus depuis Excel (peut avoir des guillemets ou *)
    raw_name = str(row[image_col]).strip().strip("'\"*")
    if not raw_name.lower().endswith('.jpg'):
        raw_name += '.jpg'

    # OCT = remplacer 'Color' par 'B-scan'
    oct_name = raw_name.replace('Color', 'B-scan')

    fundus_path = all_files.get(raw_name)
    oct_path    = all_files.get(oct_name)

    if fundus_path and oct_path:
        records.append({
            'fundus_path' : fundus_path,
            'oct_path'    : oct_path,
            'label'       : int(row['label']),
            'label_name'  : row['label_name'],
            'patient'     : str(row[subject_col]).strip() if subject_col else '',
        })
    else:
        missing.append({'fundus': raw_name, 'oct': oct_name,
                        'fundus_found': fundus_path is not None,
                        'oct_found':    oct_path is not None})

df_pairs = pd.DataFrame(records)
print(f'\n✅ Paires trouvées : {len(df_pairs)}')
print(f'❌ Manquantes     : {len(missing)}')
if missing:
    print(pd.DataFrame(missing).to_string())
print('\nDistribution :')
print(df_pairs['label_name'].value_counts())

In [ ]:
# ── Visualisation des paires ───────────────────────────────────────────────
n = min(4, len(df_pairs))
fig, axes = plt.subplots(2, n, figsize=(n * 4, 7))
fig.suptitle('Exemples de paires Fundus (haut) + OCT (bas)', fontsize=14, fontweight='bold')

for i, (_, row) in enumerate(df_pairs.head(n).iterrows()):
    img_f = mpimg.imread(row['fundus_path'])
    img_o = mpimg.imread(row['oct_path'])
    axes[0, i].imshow(img_f)
    axes[0, i].set_title(f"Fundus\n{row['label_name']} — {row['patient']}", fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(img_o, cmap='gray')
    axes[1, i].set_title(f"OCT B-scan\n{row['label_name']}", fontsize=9)
    axes[1, i].axis('off')

plt.tight_layout()
os.makedirs('/kaggle/working/outputs', exist_ok=True)
plt.savefig('/kaggle/working/outputs/pairs_visualization.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Visualisation sauvegardée')

In [ ]:
# ── Statistiques images ────────────────────────────────────────────────────
fundus_sizes, oct_sizes = [], []
for _, row in df_pairs.iterrows():
    fundus_sizes.append(Image.open(row['fundus_path']).size)
    oct_sizes.append(Image.open(row['oct_path']).size)

print('📐 Dimensions Fundus :', Counter(fundus_sizes).most_common(3))
print('📐 Dimensions OCT    :', Counter(oct_sizes).most_common(3))

# Distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
counts = df_pairs['label_name'].value_counts()
colors = ['#C00000', '#2E75B6']

axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90,
            wedgeprops={'linewidth': 2, 'edgecolor': 'white'})
axes[0].set_title('Distribution des classes', fontweight='bold')

bars = axes[1].bar(counts.index, counts.values, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.2, str(val),
                 ha='center', fontweight='bold', fontsize=13)
axes[1].set_title('Nombre de paires par classe', fontweight='bold')
axes[1].set_ylim(0, max(counts.values) + 5)

plt.tight_layout()
plt.savefig('/kaggle/working/outputs/distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sauvegarde CSV ─────────────────────────────────────────────────────────
df_pairs.to_csv('/kaggle/working/outputs/pairs_dataset.csv', index=False)

print('\n✅ EDA terminée !')
print(f'   Total paires  : {len(df_pairs)}')
print(f'   Glaucome      : {(df_pairs["label"]==1).sum()}')
print(f'   Normal        : {(df_pairs["label"]==0).sum()}')
print(f'\n📁 Sauvegardé → /kaggle/working/outputs/pairs_dataset.csv')
df_pairs.head()